In [1]:
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from torch.optim import AdamW

from evaluate import evaluate_bert_bilstm_crf
from utils.bert_bilstm_crf.data_utils import (
    read_conll_2,
    build_tag2idx,
    NERDataset,
    build_collate_fn,
)
from models.bert_bilstm_crf import BertBiLstmCrfNER


In [2]:
TRAIN_PATH = "../data/matscholar/train.txt"
VALID_PATH = "../data/matscholar/valid.txt"

MODEL_NAME = "bert-base-cased"
MAX_LENGTH = 128
BATCH_SIZE = 8
JOINT_TRAIN_EPOCHS = 8
BILSTM_ONLY_EPOCHS = 10
BERT_HIDDEN_SIZE = 768
LSTM_HIDDEN_SIZE = 256
BILSTM_OUTPUT_SIZE = LSTM_HIDDEN_SIZE * 2
DROPOUT = 0.25
BERT_LEARNING_RATE = 3e-5
OTHER_LEARNING_RATE = 1e-3
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1

TOTAL_EPOCHS = JOINT_TRAIN_EPOCHS + BILSTM_ONLY_EPOCHS
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


In [3]:
train_sentence, train_tags = read_conll_2(TRAIN_PATH)
valid_sentence, valid_tags = read_conll_2(VALID_PATH)

tag2idx, idx2tag = build_tag2idx(train_tags)


In [4]:
train_dataset = NERDataset(train_sentence, train_tags)
valid_dataset = NERDataset(valid_sentence, valid_tags)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

collate_fn = build_collate_fn(
    tokenizer=tokenizer,
    label2id=tag2idx,
    max_length=MAX_LENGTH,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
)
valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
)

model = BertBiLstmCrfNER(
    model_name=MODEL_NAME,
    num_labels=len(tag2idx),
    lstm_hidden_size=LSTM_HIDDEN_SIZE,
    dropout=DROPOUT,
    id2label=idx2tag,
    label2id=tag2idx,
    expected_bert_hidden_size=BERT_HIDDEN_SIZE,
).to(DEVICE)


def set_bert_trainable(model, trainable):
    for param in model.bert.parameters():
        param.requires_grad = trainable


def build_optimizer_and_scheduler(train_bert, num_epochs):
    param_groups = []

    if train_bert:
        param_groups.append({"params": model.bert.parameters(), "lr": BERT_LEARNING_RATE})

    param_groups.extend([
        {"params": model.bilstm.parameters(), "lr": OTHER_LEARNING_RATE},
        {"params": model.classifier.parameters(), "lr": OTHER_LEARNING_RATE},
        {"params": model.crf.parameters(), "lr": OTHER_LEARNING_RATE},
    ])

    optimizer = AdamW(param_groups, weight_decay=WEIGHT_DECAY)

    total_steps = len(train_loader) * num_epochs
    warmup_steps = int(WARMUP_RATIO * total_steps)
    scheduler = get_linear_schedule_with_warmup(
        optimizer=optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )

    return optimizer, scheduler


set_bert_trainable(model, True)
optimizer, scheduler = build_optimizer_and_scheduler(
    train_bert=True,
    num_epochs=JOINT_TRAIN_EPOCHS,
)
bert_is_trainable = True


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
best_valid_f1 = 0.0

for epoch in range(TOTAL_EPOCHS):
    if epoch == JOINT_TRAIN_EPOCHS and bert_is_trainable:
        set_bert_trainable(model, False)
        optimizer, scheduler = build_optimizer_and_scheduler(
            train_bert=False,
            num_epochs=BILSTM_ONLY_EPOCHS,
        )
        bert_is_trainable = False
        print("\nSwitch to stage 2: freeze BERT and train BiLSTM, classifier, and CRF.")

    model.train()

    if not bert_is_trainable:
        model.bert.eval()

    total_loss = 0.0

    for batch in train_loader:
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        word_attention_mask = batch["word_attention_mask"].to(DEVICE)
        first_subword_positions = batch["first_subword_positions"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        token_type_ids = batch.get("token_type_ids")

        if token_type_ids is not None:
            token_type_ids = token_type_ids.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            word_attention_mask=word_attention_mask,
            first_subword_positions=first_subword_positions,
            labels=labels,
            token_type_ids=token_type_ids,
        )

        loss = outputs["loss"]
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

    train_loss = total_loss / len(train_loader)

    valid_loss, valid_precision, valid_recall, valid_f1, _, _ = evaluate_bert_bilstm_crf(
        model=model,
        dataloader=valid_loader,
        id2label=idx2tag,
        device=DEVICE,
    )

    stage_name = "stage 1 joint training" if bert_is_trainable else "stage 2 frozen BERT"

    print(f"\nEpoch {epoch + 1}/{TOTAL_EPOCHS} - {stage_name}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Valid Loss: {valid_loss:.4f}")
    print(f"Valid Precision: {valid_precision:.4f}")
    print(f"Valid Recall: {valid_recall:.4f}")
    print(f"Valid F1: {valid_f1:.4f}")

    if valid_f1 > best_valid_f1:
        best_valid_f1 = valid_f1

        torch.save({
            "model": model.state_dict(),
            "model_type": "bert_bilstm_crf",
            "architecture": "tokenizer_bert_bilstm_crf",
            "tag2idx": tag2idx,
            "idx2tag": idx2tag,
            "model_name": MODEL_NAME,
            "max_length": MAX_LENGTH,
            "bert_hidden_size": BERT_HIDDEN_SIZE,
            "lstm_hidden_size": LSTM_HIDDEN_SIZE,
            "bilstm_output_size": BILSTM_OUTPUT_SIZE,
            "dropout": DROPOUT,
            "joint_train_epochs": JOINT_TRAIN_EPOCHS,
            "bilstm_only_epochs": BILSTM_ONLY_EPOCHS,
            "bert_learning_rate": BERT_LEARNING_RATE,
            "other_learning_rate": OTHER_LEARNING_RATE,
            "best_stage": stage_name,
        }, "../models/bert_bilstm_crf.pt")

        print("Saved best model")



Epoch 1/35 - stage 1 joint training
Train Loss: 13.6489
Valid Loss: 5.7107
Valid Precision: 0.7720
Valid Recall: 0.8066
Valid F1: 0.7889
Saved best model

Epoch 2/35 - stage 1 joint training
Train Loss: 3.7339
Valid Loss: 5.0965
Valid Precision: 0.8219
Valid Recall: 0.8462
Valid F1: 0.8338
Saved best model

Epoch 3/35 - stage 1 joint training
Train Loss: 1.7894
Valid Loss: 4.8561
Valid Precision: 0.8378
Valid Recall: 0.8601
Valid F1: 0.8488
Saved best model

Epoch 4/35 - stage 1 joint training
Train Loss: 0.8861
Valid Loss: 6.2569
Valid Precision: 0.8261
Valid Recall: 0.8653
Valid F1: 0.8453

Epoch 5/35 - stage 1 joint training
Train Loss: 0.4316
Valid Loss: 6.7254
Valid Precision: 0.8392
Valid Recall: 0.8688
Valid F1: 0.8537
Saved best model

Switch to stage 2: freeze BERT and train BiLSTM, classifier, and CRF.

Epoch 6/35 - stage 2 frozen BERT
Train Loss: 0.1436
Valid Loss: 7.5190
Valid Precision: 0.8377
Valid Recall: 0.8635
Valid F1: 0.8504

Epoch 7/35 - stage 2 frozen BERT
Train L